

















































































































































































# 🎁 Bônus 2 da Semana 11 — ETL com Python e PostgreSQL: Bronze, Prata e Ouro

Este é outro bônus opcional, separado do `notebook_bonus_postgresql.ipynb` — você pode fazer os dois, em qualquer ordem, mas ambos seguem a mesma lógica de conexão com o PostgreSQL (`psycopg2`) e o mesmo banco `northwind`.

Aqui você vai:
1. Praticar CRUD numa tabela nova (`suppliers`), diferente da que já foi usada no primeiro bônus (`customers`).
2. Simular, de verdade, um pipeline de ETL organizado em 3 camadas — **Bronze, Prata e Ouro** —, escrevendo arquivos `.csv` reais dentro de uma pasta `etl_medalhao/` no projeto, pra você abrir no Explorer do VS Code e ver a estrutura de pastas se formando, exatamente como acontece num pipeline de dados de verdade.

**⚠️ Pré-requisito:** o PostgreSQL precisa estar rodando na sua máquina, e o banco `northwind` já precisa existir (criado nas Semanas 08/09). Troque `SUA_SENHA_AQUI` pela senha que você mesmo configurou no PostgreSQL.

## Connect — a mesma conexão de sempre

Se você já fez o primeiro bônus, isso vai parecer repetido — e é proposital: toda vez que um script novo roda, ele precisa abrir sua própria conexão. Se você ainda não fez o primeiro bônus, aqui vai o resumo: `psycopg2.connect(...)` abre a conexão com o PostgreSQL, e `.cursor()` cria o objeto que executa os comandos SQL.

In [ ]:
import psycopg2

conexao = psycopg2.connect(
    host="localhost",
    dbname="northwind",
    user="postgres",
    password="SUA_SENHA_AQUI",
)
cursor = conexao.cursor()
print("Conexão bem sucedida")

## CRUD numa tabela nova: `suppliers`

**Contextualização:** a Northwind está expandindo as compras no Brasil, e vai cadastrar um fornecedor brasileiro novo de especiarias.

A tabela `suppliers` guarda os fornecedores da Northwind — mesma ideia de `customers`, só que do outro lado da cadeia (quem vende PRA Northwind, não quem compra DELA). Repare que `supplier_id` também não é gerado automaticamente pelo banco (como `customer_id` no primeiro bônus, é um valor que você escolhe na hora do cadastro) — antes de inserir, é preciso saber qual é o maior id já usado, pra não repetir um que já existe.

O comando abaixo descobre isso com a função `MAX()`, que retorna o maior valor já cadastrado numa coluna.

In [ ]:
cursor.execute("SELECT MAX(supplier_id) FROM suppliers")
maior_id = cursor.fetchall()[0][0]
print("Maior supplier_id atual:", maior_id)

**Create:** cadastrando o novo fornecedor, com `supplier_id` igual ao maior id atual + 1.

In [ ]:
novo_id = maior_id + 1

cursor.execute(
    "INSERT INTO suppliers (supplier_id, company_name, contact_name, country, phone) VALUES (%s, %s, %s, %s, %s)",
    (novo_id, "Nordeste Especiarias Ltda", "Maria Oliveira", "Brazil", "(84) 3232-1000"),
)
conexao.commit()

cursor.execute("SELECT supplier_id, company_name, country FROM suppliers WHERE supplier_id = %s", (novo_id,))
print(cursor.fetchall())

**Read:** consultando todos os fornecedores do Brasil — reforçando o `WHERE` que você já usou no primeiro bônus, agora numa tabela diferente.

In [ ]:
cursor.execute("SELECT supplier_id, company_name, country FROM suppliers WHERE country = %s", ("Brazil",))
print(cursor.fetchall())

**Update:** a Nordeste Especiarias avisou o site da empresa.

In [ ]:
cursor.execute(
    "UPDATE suppliers SET homepage = %s WHERE supplier_id = %s",
    ("www.nordesteespeciarias.com.br", novo_id),
)
conexao.commit()

cursor.execute("SELECT supplier_id, company_name, homepage FROM suppliers WHERE supplier_id = %s", (novo_id,))
print(cursor.fetchall())

**Delete:** o cadastro foi só um teste — hora de remover.

In [ ]:
cursor.execute("DELETE FROM suppliers WHERE supplier_id = %s", (novo_id,))
conexao.commit()

cursor.execute("SELECT * FROM suppliers WHERE supplier_id = %s", (novo_id,))
print("Deve ficar vazio:", cursor.fetchall())

## Simulando um ETL de verdade — Bronze, Prata e Ouro

Esse padrão de organizar dados em 3 camadas — **Bronze** (dado bruto, exatamente como veio da fonte), **Prata** (dado já limpo/filtrado) e **Ouro** (dado agregado, pronto pra virar relatório ou dashboard) — é chamado de **arquitetura medalhão**, e é usado de verdade em empresas que trabalham com dados em grande escala.

Você vai construir as 3 camadas com consultas SQL diferentes contra o mesmo `northwind`, salvando cada resultado num arquivo `.csv` dentro de uma pasta própria — `etl_medalhao/bronze/`, `etl_medalhao/prata/` e `etl_medalhao/ouro/`. Depois de rodar, abra o Explorer do VS Code (barra lateral esquerda) e navegue até a pasta `etl_medalhao/`, criada dentro da pasta desta semana — as 3 subpastas vão aparecer ali, cada uma com seu arquivo, exatamente como um pipeline de dados de verdade organiza as camadas.

**Simplificação proposital:** num pipeline de produção real, a camada Prata normalmente leria o arquivo da Bronze (em vez de consultar o banco de novo), e a Ouro leria o arquivo da Prata. Aqui, pra manter o foco na integração Python + SQL, cada camada faz sua própria consulta SQL — mas o princípio de "cada camada é mais refinada que a anterior" é o mesmo.

### 🥉 Bronze — o dado bruto, sem filtro nenhum

**Contextualização:** a Northwind quer guardar uma cópia bruta de tudo que já foi vendido, sem excluir nada — nem os pedidos ainda não despachados, nem os produtos descontinuados. É esse dado bruto que vira o ponto de partida do pipeline.

In [ ]:
import os
import pandas as pd

bronze = pd.read_sql('''
    SELECT o.order_id, o.order_date, o.shipped_date,
           c.company_name AS cliente, c.country AS pais_cliente,
           p.product_name AS produto, p.discontinued,
           od.unit_price, od.quantity, od.discount
    FROM order_details od
    JOIN orders o ON o.order_id = od.order_id
    JOIN customers c ON c.customer_id = o.customer_id
    JOIN products p ON p.product_id = od.product_id
''', conexao)

os.makedirs("etl_medalhao/bronze", exist_ok=True)
bronze.to_csv("etl_medalhao/bronze/vendas_bruto.csv", index=False)

confirmacao = pd.read_csv("etl_medalhao/bronze/vendas_bruto.csv")
display(confirmacao.head(10))
print(f"{len(confirmacao)} linhas salvas em etl_medalhao/bronze/vendas_bruto.csv")

Duas funções novas nesse código: `os.makedirs("etl_medalhao/bronze", exist_ok=True)` cria a pasta se ela ainda não existir (o `exist_ok=True` evita erro caso ela já exista, de uma rodada anterior); `bronze.to_csv(..., index=False)` salva o DataFrame em disco como um arquivo `.csv` de verdade, sem a coluna de índice numérico que o Pandas usa internamente. Reabrir esse mesmo arquivo logo depois com `pd.read_csv()` é o que garante que ele foi realmente criado — não basta confiar que "rodou sem erro".

### 🥈 Prata — o dado já limpo

**Contextualização:** agora a equipe de dados precisa de uma versão só com pedidos que já chegaram de verdade, e só com produtos que a Northwind ainda vende — a mesma limpeza que você já viu no primeiro bônus (`shipped_date IS NOT NULL`, `discontinued = 0`), só que agora ela decide o que entra na camada Prata.

In [ ]:
prata = pd.read_sql('''
    SELECT o.order_id, o.order_date, o.shipped_date,
           c.company_name AS cliente, c.country AS pais_cliente,
           p.product_name AS produto,
           od.unit_price, od.quantity, od.discount
    FROM order_details od
    JOIN orders o ON o.order_id = od.order_id
    JOIN customers c ON c.customer_id = o.customer_id
    JOIN products p ON p.product_id = od.product_id
    WHERE o.shipped_date IS NOT NULL
      AND p.discontinued = 0
''', conexao)

os.makedirs("etl_medalhao/prata", exist_ok=True)
prata.to_csv("etl_medalhao/prata/vendas_limpo.csv", index=False)

confirmacao = pd.read_csv("etl_medalhao/prata/vendas_limpo.csv")
display(confirmacao.head(10))
print(f"Bronze tinha {len(bronze)} linhas; Prata ficou com {len(confirmacao)} linhas depois da limpeza.")

### 🥇 Ouro — o dado agregado, pronto pra virar relatório

**Contextualização:** a diretoria não quer ver linha por linha — quer um resumo direto: quanto cada cliente já gastou, quantos pedidos fez, e qual o valor médio dos itens comprados, pra decidir quem são os clientes mais importantes.

Uma novidade na consulta abaixo: `COUNT(DISTINCT o.order_id)`. Sem o `DISTINCT`, o `COUNT` contaria uma vez PARA CADA ITEM do pedido (porque o `JOIN` com `order_details` multiplica uma linha de pedido em várias, uma por produto comprado) — o `DISTINCT` garante que cada `order_id` seja contado só uma vez, mesmo que ele apareça repetido no resultado do JOIN.

In [ ]:
ouro = pd.read_sql('''
    SELECT c.company_name AS cliente,
           COUNT(DISTINCT o.order_id) AS total_pedidos,
           ROUND(SUM(od.unit_price * od.quantity * (1 - od.discount))::numeric, 2) AS receita_total,
           ROUND(AVG(od.unit_price * od.quantity * (1 - od.discount))::numeric, 2) AS ticket_medio_item
    FROM order_details od
    JOIN orders o ON o.order_id = od.order_id
    JOIN customers c ON c.customer_id = o.customer_id
    WHERE o.shipped_date IS NOT NULL
    GROUP BY c.company_name
    ORDER BY receita_total DESC
    LIMIT 10
''', conexao)

os.makedirs("etl_medalhao/ouro", exist_ok=True)
ouro.to_csv("etl_medalhao/ouro/receita_por_cliente.csv", index=False)

confirmacao = pd.read_csv("etl_medalhao/ouro/receita_por_cliente.csv")
display(confirmacao)

### 📊 Visualizando a camada Ouro

A camada Ouro já está pronta pra virar um gráfico de verdade — é exatamente pra isso que ela existe. Com o `matplotlib`, que você já viu na Semana 07, dá pra transformar a tabela `ouro` (que acabou de ser salva) num gráfico de barras da receita por cliente.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.barh(ouro["cliente"], ouro["receita_total"])
plt.xlabel("Receita total (R$)")
plt.title("Receita por cliente — camada Ouro")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Conferindo a estrutura de pastas criada

O código abaixo percorre a pasta `etl_medalhao/` e imprime tudo que foi criado — mas o mais importante mesmo é você abrir o Explorer do VS Code e ver isso com os próprios olhos.

In [ ]:
for raiz, pastas, arquivos in os.walk("etl_medalhao"):
    nivel = raiz.replace("etl_medalhao", "").count(os.sep)
    indentacao = "  " * nivel
    print(f"{indentacao}{os.path.basename(raiz) or 'etl_medalhao'}/")
    for arquivo in sorted(arquivos):
        print(f"{indentacao}  {arquivo}")

In [ ]:
cursor.close()
conexao.close()

## ✏️ Atividade Bônus 1 — Sua vez (camada Prata)

**Contextualização:** o time comercial quer uma versão da camada Prata só com as vendas pro Brasil, pra analisar separado do resto do mundo.

**Comando:** conecte no banco `northwind`, escreva uma consulta parecida com a da camada Prata que você acabou de ver, mas filtrando também `c.country = 'Brazil'` (além do `shipped_date IS NOT NULL` e `discontinued = 0`). Salve o resultado em `etl_medalhao/prata/vendas_brasil.csv`, e confirme reabrindo o arquivo com `pd.read_csv()` e `display()`. Feche a conexão ao final.

In [ ]:
# escreva seu código aqui

## ✏️ Atividade Bônus 2 — Sua vez (camada Ouro)

**Contextualização:** a diretoria agora quer saber a receita por categoria de produto, dentro da camada Ouro — igual ao que uma diretoria de verdade pediria pra decidir onde investir.

**Comando:** conecte no banco `northwind`, escreva uma consulta que junte `order_details`, `products` e `categories`, agrupando por `category_name` e somando a receita (mesma fórmula de sempre: `unit_price * quantity * (1 - discount)`). Salve o resultado em `etl_medalhao/ouro/receita_por_categoria.csv`, confirme reabrindo o arquivo, e feche a conexão.

In [ ]:
# escreva seu código aqui

### ✅ O que você fez neste bônus

- Praticou CRUD numa tabela nova (`suppliers`), incluindo descobrir o próximo id disponível com `MAX()`.
- Entendeu o conceito de arquitetura medalhão: Bronze (dado bruto), Prata (dado limpo) e Ouro (dado agregado).
- Escreveu 3 arquivos `.csv` reais, organizados em pastas, dentro do próprio projeto — e viu a estrutura no Explorer do VS Code.
- Usou `COUNT(DISTINCT ...)` pra contar pedidos únicos apesar do JOIN multiplicar linhas.
- Transformou a camada Ouro num gráfico de verdade com `matplotlib`, reaproveitando o que já tinha visto na Semana 07.
- Reaproveitou a limpeza de dados (`shipped_date IS NOT NULL`, `discontinued = 0`) e a fórmula de receita (`unit_price * quantity * (1 - discount)`) já vistas no primeiro bônus.